# Módulo 3 — Visualização de Dados
### Curso Introdutório de Python para Ciência de Dados
**Disciplina:** T326 - Ciência de Dados | UNIFOR  
**Dataset:** Brazilian Cities — 5.578 municípios brasileiros

---

## 📚 O que você vai aprender neste módulo

| Seção | Conteúdo |
|-------|----------|
| 3.1 | Fundamentos de visualização — por que e quando usar cada gráfico |
| 3.2 | Matplotlib — gráficos base |
| 3.3 | Seaborn — visualizações estatísticas |
| 3.4 | Exemplos práticos reais com o dataset |
| 3.5 | Exercícios |

> 🎨 **Boa visualização** = dado certo + gráfico certo + design limpo


---
## Configurações e Carregamento dos Dados

In [ ]:
# ── Importações ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, io

warnings.filterwarnings('ignore')

# Configurações globais de estilo
plt.rcParams.update({
    'figure.dpi':       120,
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F9FA',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.family':      'DejaVu Sans',
})
sns.set_palette('Set2')

print(f"✅ Matplotlib {plt.matplotlib.__version__}")
print(f"✅ Seaborn    {sns.__version__}")


In [ ]:
# ── Carregamento do dataset ──────────────────────────────────────
def carregar_brazilian_cities(caminho):
    with open(caminho, 'r', encoding='utf-8', newline='') as f:
        raw = f.read()
    linhas = raw.split('\r\n')
    def fix(l):
        if l.startswith('"') and l.endswith('"'):
            l = l[1:-1]
        return l.replace('""', '"')
    conteudo = '\n'.join([fix(l) for l in linhas if l.strip()])
    return pd.read_csv(io.StringIO(conteudo))

df = carregar_brazilian_cities('../datasets/brazilian_city.csv')

# Preparação rápida dos dados
df = df.rename(columns={
    'IDHM Ranking 2010': 'IDHM_Ranking',
    'IBGE_CROP_PRODUCTION_$': 'IBGE_CROP_PROD',
    'WAL-MART': 'WALMART', 'IBGE_1-4': 'IBGE_1a4',
    'IBGE_5-9': 'IBGE_5a9', 'IBGE_10-14': 'IBGE_10a14',
    'IBGE_15-59': 'IBGE_15a59', 'IBGE_60+': 'IBGE_60mais',
})
df['POP_FINAL'] = df['IBGE_RES_POP'].where(df['IBGE_RES_POP'] > 0, df['ESTIMATED_POP'])
df['DENSIDADE'] = np.where(df['AREA'] > 0, (df['POP_FINAL'] / df['AREA']).round(2), np.nan)
df['TIPO'] = df['CAPITAL'].map({1: 'Capital', 0: 'Interior'})
df['IDHM_FAIXA'] = pd.cut(df['IDHM'],
    bins=[0, 0.499, 0.599, 0.699, 0.799, 1.0],
    labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto'])

print(f"✅ Dataset pronto: {df.shape[0]:,} municípios × {df.shape[1]} variáveis")


---
## 3.1 Teoria — Fundamentos de Visualização

### Qual gráfico usar?

| Pergunta | Tipo de Gráfico |
|----------|----------------|
| Distribuição de uma variável? | Histograma, KDE, Boxplot |
| Relação entre duas variáveis? | Scatter plot |
| Comparação entre categorias? | Barras, Boxplot agrupado |
| Evolução ao longo do tempo? | Linha |
| Parte de um todo? | Pizza (use com cuidado) |
| Correlação entre muitas variáveis? | Heatmap |

### Boas práticas
- ✅ Título descritivo
- ✅ Rótulos nos eixos (com unidade)
- ✅ Fonte dos dados
- ✅ Paleta acessível (daltônicos)
- ❌ Evite 3D desnecessário
- ❌ Evite excesso de informação em um gráfico


---
## 3.2 Matplotlib — A Base das Visualizações

**Matplotlib** é a biblioteca fundamental. Entender sua anatomia é essencial:

```
Figure  →  a "tela" inteira
  └── Axes  →  o "painel" do gráfico
        ├── Title
        ├── X/Y Axis (com Label e Ticks)
        └── Plot (barras, linha, pontos...)
```


In [ ]:
# ── 3.2.1 Anatomia de um gráfico Matplotlib ─────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

# Dados de exemplo
x = [1, 2, 3, 4, 5]
y = [3, 7, 2, 9, 4]

ax.plot(x, y, marker='o', color='steelblue', linewidth=2, markersize=8, label='Série A')
ax.set_title('Anatomia de um Gráfico Matplotlib', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Eixo X (unidade)', fontsize=11)
ax.set_ylabel('Eixo Y (unidade)', fontsize=11)
ax.legend()

# Anotações didáticas
ax.annotate('Linha (plot)', xy=(3, 2), xytext=(3.5, 1.5),
            arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=9)
ax.annotate('Marcador', xy=(4, 9), xytext=(2.5, 8.5),
            arrowprops=dict(arrowstyle='->', color='green'), color='green', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.2.2 Histograma — Distribuição do IDHM ─────────────────────
# Histograma mostra como os dados se distribuem
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(df['IDHM'].dropna(), bins=40, color='#4C72B0', edgecolor='white',
        alpha=0.85, rwidth=0.9)

# Linhas de referência: média e mediana
media = df['IDHM'].mean()
mediana = df['IDHM'].median()
ax.axvline(media,   color='#DD4949', linestyle='--', linewidth=2, label=f'Média = {media:.3f}')
ax.axvline(mediana, color='#2CA02C', linestyle='-.',  linewidth=2, label=f'Mediana = {mediana:.3f}')

ax.set_title('Distribuição do IDHM nos Municípios Brasileiros', fontsize=14, fontweight='bold')
ax.set_xlabel('IDHM', fontsize=12)
ax.set_ylabel('Número de Municípios', fontsize=12)
ax.legend(fontsize=11)
ax.text(0.98, 0.95, f'n = {df["IDHM"].notna().sum():,} municípios',
        transform=ax.transAxes, ha='right', va='top', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print(f"IDHM — Média: {media:.3f} | Mediana: {mediana:.3f} | Desvio: {df['IDHM'].std():.3f}")


In [ ]:
# ── 3.2.3 Gráfico de Barras — Municípios por Estado ─────────────
# Contagem de municípios por UF (ordenado)
munic_por_estado = df.groupby('STATE')['CITY'].count().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))

cores = ['#E06C75' if v == munic_por_estado.max() else '#4C72B0' for v in munic_por_estado]
bars = ax.bar(munic_por_estado.index, munic_por_estado.values, color=cores, edgecolor='white')

# Rótulos sobre as barras
for bar, val in zip(bars, munic_por_estado.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_title('Número de Municípios por Estado', fontsize=14, fontweight='bold')
ax.set_xlabel('Estado (UF)', fontsize=12)
ax.set_ylabel('Quantidade de Municípios', fontsize=12)
ax.axhline(munic_por_estado.mean(), color='red', linestyle='--', alpha=0.6,
           label=f'Média = {munic_por_estado.mean():.0f}')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.2.4 Scatter Plot — PIB per Capita × IDHM ──────────────────
# Relação entre desenvolvimento humano e riqueza
df_plot = df[(df['GDP_CAPITA'] > 0) & (df['GDP_CAPITA'] < 200_000)].copy()

fig, ax = plt.subplots(figsize=(10, 6))

# Colorir por tipo (capital vs interior)
cores_tipo = {'Capital': '#E06C75', 'Interior': '#4C72B0'}
for tipo, grupo in df_plot.groupby('TIPO'):
    ax.scatter(grupo['IDHM'], grupo['GDP_CAPITA'],
               alpha=0.4, s=20, color=cores_tipo[tipo], label=tipo)

# Linha de tendência (regressão linear)
z = np.polyfit(df_plot['IDHM'].dropna(), df_plot['GDP_CAPITA'].dropna(), 1)
p = np.poly1d(z)
x_linha = np.linspace(df_plot['IDHM'].min(), df_plot['IDHM'].max(), 100)
ax.plot(x_linha, p(x_linha), 'k--', alpha=0.7, linewidth=2, label='Tendência')

ax.set_title('IDHM × PIB per Capita nos Municípios Brasileiros', fontsize=14, fontweight='bold')
ax.set_xlabel('IDHM', fontsize=12)
ax.set_ylabel('PIB per Capita (R$)', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.2.5 Subplots — Múltiplos gráficos em uma figura ───────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribuições de Variáveis Socioeconômicas', fontsize=15, fontweight='bold', y=1.02)

# Painel 1: IDHM
axes[0].hist(df['IDHM'].dropna(), bins=30, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].set_title('IDHM')
axes[0].set_xlabel('IDHM')
axes[0].set_ylabel('Municípios')

# Painel 2: GDP_CAPITA (log scale)
gdp_valido = df['GDP_CAPITA'][df['GDP_CAPITA'] > 0]
axes[1].hist(np.log10(gdp_valido), bins=30, color='#55A868', edgecolor='white', alpha=0.85)
axes[1].set_title('PIB per Capita (escala log)')
axes[1].set_xlabel('log₁₀(PIB per Capita)')
axes[1].set_ylabel('Municípios')

# Painel 3: Densidade demográfica (log scale)
dens_valido = df['DENSIDADE'][df['DENSIDADE'] > 0]
axes[2].hist(np.log10(dens_valido), bins=30, color='#C44E52', edgecolor='white', alpha=0.85)
axes[2].set_title('Densidade Demográfica (escala log)')
axes[2].set_xlabel('log₁₀(hab/km²)')
axes[2].set_ylabel('Municípios')

for ax in axes:
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


---
## 3.3 Seaborn — Visualizações Estatísticas

**Seaborn** é construído sobre Matplotlib e especializado em gráficos estatísticos. Principais vantagens:
- Gráficos mais bonitos com menos código
- Integração nativa com DataFrames
- Suporte automático a grupos (hue, palette)


In [ ]:
# ── 3.3.1 Boxplot — IDHM por Região ─────────────────────────────
# Definir macro-regiões
regioes = {
    'Norte':     ['AM','PA','RR','RO','AC','AP','TO'],
    'Nordeste':  ['MA','PI','CE','RN','PB','PE','AL','SE','BA'],
    'Centro-Oeste': ['MT','MS','GO','DF'],
    'Sudeste':   ['SP','RJ','MG','ES'],
    'Sul':       ['PR','SC','RS']
}
estado_regiao = {uf: reg for reg, ufs in regioes.items() for uf in ufs}
df['REGIAO'] = df['STATE'].map(estado_regiao)

ordem = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
palette = ['#E07B54', '#E0C354', '#54A0E0', '#54E089', '#9B54E0']

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df, x='REGIAO', y='IDHM', order=ordem,
            palette=palette, width=0.6, fliersize=2, ax=ax)

ax.set_title('Distribuição do IDHM por Região Brasileira', fontsize=14, fontweight='bold')
ax.set_xlabel('Região', fontsize=12)
ax.set_ylabel('IDHM', fontsize=12)
ax.axhline(df['IDHM'].median(), color='gray', linestyle='--', alpha=0.6, label='Mediana Nacional')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3.2 Violinplot — Distribuição detalhada ───────────────────
fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(data=df, x='REGIAO', y='IDHM', order=ordem, palette=palette, inner='quartile', ax=ax)

ax.set_title('Distribuição Detalhada do IDHM por Região (Linhas internas = quartis)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Região', fontsize=12)
ax.set_ylabel('IDHM', fontsize=12)

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3.3 Heatmap — Matriz de Correlação ────────────────────────
# Selecionar variáveis para correlação
vars_corr = ['IDHM', 'GDP_CAPITA', 'IDHM_Renda', 'IDHM_Longevidade',
             'IDHM_Educacao', 'DENSIDADE', 'PAY_TV', 'FIXED_PHONES', 'Cars']

corr_matrix = df[vars_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # mascarar triângulo superior

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlação de Pearson'}, ax=ax)

ax.set_title('Matriz de Correlação — Variáveis Socioeconômicas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3.4 Barplot com erro padrão — IDHM médio por região ────────
fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(data=df, x='REGIAO', y='IDHM', order=ordem,
            palette=palette, capsize=0.1, errorbar='se', ax=ax)

# Adicionar valores sobre as barras
for patch in ax.patches:
    h = patch.get_height()
    if h > 0:
        ax.text(patch.get_x() + patch.get_width()/2, h + 0.002,
                f'{h:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('IDHM Médio por Região Brasileira',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Região', fontsize=12)
ax.set_ylabel('IDHM Médio', fontsize=12)
ax.set_ylim(0.5, 0.85)

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.3.5 Pairplot — Relações entre variáveis (amostra) ──────────
# Amostra de 500 cidades para não sobrecarregar
df_amostra = df[['IDHM', 'GDP_CAPITA', 'IDHM_Renda', 'IDHM_Educacao', 'REGIAO']].dropna()
df_amostra = df_amostra[df_amostra['GDP_CAPITA'] < 200_000].sample(500, random_state=42)

g = sns.pairplot(df_amostra, hue='REGIAO', hue_order=ordem,
                 palette=dict(zip(ordem, palette)),
                 diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20})
g.fig.suptitle('Relações entre Variáveis — Amostra de 500 Municípios', y=1.02,
               fontsize=14, fontweight='bold')
plt.show()


---
## 3.4 Exemplos Práticos Reais

Análises completas com narrativa de dados — o tipo de gráfico que aparece em relatórios e apresentações reais.


In [ ]:
# ── 3.4.1 Dashboard: Top 15 estados por IDHM médio ─────────────
idhm_estado = df.groupby('STATE').agg(
    IDHM_Medio=('IDHM', 'mean'),
    N_Munic=('CITY', 'count'),
    Pop_Total=('POP_FINAL', 'sum')
).sort_values('IDHM_Medio', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))
cores_barra = ['#E07B54' if v < 0.65 else '#55A868' if v >= 0.72 else '#4C72B0'
               for v in idhm_estado['IDHM_Medio']]

barras = ax.barh(idhm_estado.index, idhm_estado['IDHM_Medio'],
                 color=cores_barra, edgecolor='white', height=0.7)

for bar, val in zip(barras, idhm_estado['IDHM_Medio']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9, fontweight='bold')

ax.axvline(df['IDHM'].mean(), color='black', linestyle='--',
           alpha=0.7, label=f'Média Brasil = {df["IDHM"].mean():.3f}')
ax.set_title('IDHM Médio por Estado (🟢 Alto  🔵 Médio  🟠 Baixo)', fontsize=14, fontweight='bold')
ax.set_xlabel('IDHM Médio', fontsize=12)
ax.legend()
ax.set_xlim(0.57, 0.82)

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.4.2 Análise: Composição do PIB por Região ─────────────────
gva_regiao = df.groupby('REGIAO')[['GVA_AGROPEC', 'GVA_INDUSTRY', 'GVA_SERVICES']].sum()
gva_pct = gva_regiao.div(gva_regiao.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
categorias = ['Agropecuária', 'Indústria', 'Serviços']
cores_gva = ['#55A868', '#4C72B0', '#E07B54']

bottom = np.zeros(len(ordem))
gva_pct_ord = gva_pct.reindex(ordem)

for i, (cat, cor) in enumerate(zip(['GVA_AGROPEC', 'GVA_INDUSTRY', 'GVA_SERVICES'], cores_gva)):
    valores = gva_pct_ord[cat].values
    bars = ax.bar(gva_pct_ord.index, valores, bottom=bottom,
                  color=cor, label=categorias[i], edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, valores):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}%', ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')
    bottom += valores

ax.set_title('Composição do VAB (Valor Adicionado Bruto) por Região (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Participação (%)', fontsize=12)
ax.set_ylim(0, 100)
ax.legend(loc='upper right', framealpha=0.9)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.show()


In [ ]:
# ── 3.4.3 Mapa de calor: IDHM médio por Estado × Componente ──────
idhm_comp = df.groupby('STATE')[['IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao']].mean()
idhm_comp.columns = ['IDH Geral', 'Renda', 'Longevidade', 'Educação']
idhm_comp = idhm_comp.sort_values('IDH Geral', ascending=False)

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(idhm_comp, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0.55, vmax=0.90, linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Valor do IDH'})
ax.set_title('Componentes do IDHM por Estado (ordenado pelo IDH Geral)', fontsize=14, fontweight='bold')
ax.set_ylabel('Estado', fontsize=12)

plt.tight_layout()
plt.show()


---
## 3.5 Exercícios Práticos — Módulo 3

> O **gabarito** está no arquivo `gabarito_exercicios.ipynb`.

---

### Exercício 1 — Histograma
Crie um histograma do **PIB per capita** (somente valores > 0 e < 150.000). Adicione linhas de média e mediana.

```python
# Seu código aqui
```

---

### Exercício 2 — Barras horizontais
Crie um gráfico de barras horizontais com as **10 cidades mais populosas** do Brasil. Inclua a população como rótulo.

```python
# Seu código aqui
```

---

### Exercício 3 — Boxplot com Seaborn
Compare a **distribuição do GDP_CAPITA** entre cidades **Urbanas**, **Rural Adjacente** e **Rural Remoto** usando boxplot do Seaborn.

```python
# Seu código aqui
```

---

### Exercício 4 — Scatter Plot
Crie um scatter plot de **ÁREA × POP_FINAL** (escala logarítmica nos dois eixos), colorindo por Região.

```python
# Seu código aqui
```

---

### Exercício 5 — Visualização livre
Escolha **duas variáveis** do dataset e crie uma visualização que conte uma história sobre as cidades brasileiras. Adicione título e legendas adequados.

```python
# Seu código aqui
```


In [ ]:
# Espaço para suas respostas

# Exercício 1

# Exercício 2

# Exercício 3

# Exercício 4

# Exercício 5


---
## 📌 Resumo — Comandos Essenciais de Visualização

### Matplotlib
```python
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(data, bins=30)                     # histograma
ax.bar(x, y)                               # barras verticais
ax.barh(y, x)                              # barras horizontais
ax.scatter(x, y, c=cores, s=tamanho)       # dispersão
ax.plot(x, y, linestyle='--')              # linha
ax.set_title('Título')
ax.set_xlabel('X'); ax.set_ylabel('Y')
ax.legend()
plt.tight_layout(); plt.show()
```

### Seaborn
```python
sns.histplot(data=df, x='col', hue='grupo')
sns.boxplot(data=df, x='cat', y='num')
sns.violinplot(data=df, x='cat', y='num')
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm')
sns.barplot(data=df, x='cat', y='num', errorbar='se')
sns.scatterplot(data=df, x='x', y='y', hue='grupo', size='tamanho')
sns.pairplot(df, hue='grupo')
```

---
*Próximo: **Módulo 4 — Análise Exploratória de Dados (EDA)***
